In [0]:
# =============================================================
# run_framework.py — Medallion Architecture (L0 / L1 / L2)
# Reads config from control tables and processes all layers
# =============================================================

In [0]:
dbutils.widgets.text("GROUP_ID", "")
dbutils.widgets.text("RUN_LAYER", "ALL")   # ALL / L0 / L1 / L2

group_id  = dbutils.widgets.get("GROUP_ID").strip().upper()
run_layer = dbutils.widgets.get("RUN_LAYER").strip().upper()

if not group_id:
    raise Exception("GROUP_ID is empty.")

CATALOG = "demo_catalog"

# ── Auto-detect layer from GROUP_ID suffix ────────────────────
# FINANCE_MEDALLION_L0 → L0, FINANCE_MEDALLION_L1 → L1, etc.
# RUN_LAYER widget overrides this if set to something other than ALL
if run_layer == "ALL":
    if group_id.endswith("_L0"):
        run_layer = "L0"
    elif group_id.endswith("_L1"):
        run_layer = "L1"
    elif group_id.endswith("_L2"):
        run_layer = "L2"

print(f"GROUP_ID   : {group_id}")
print(f"RUN_LAYER  : {run_layer} (auto-detected from GROUP_ID suffix)")

In [0]:
import pandas as pd
import requests
import io
import traceback
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

In [0]:
# ── Helpers ───────────────────────────────────────────────────

def is_http(url):
    return url.strip().lower().startswith("http")

def read_source(source_url, file_format):
    fmt = file_format.strip().lower()
    if is_http(source_url):
        resp = requests.get(source_url, timeout=60)
        resp.raise_for_status()
        raw  = resp.content
        if fmt == "csv":
            return spark.createDataFrame(pd.read_csv(io.BytesIO(raw)))
        elif fmt == "json":
            return spark.createDataFrame(pd.read_json(io.BytesIO(raw)))
        elif fmt in ("parquet",):
            return spark.createDataFrame(pd.read_parquet(io.BytesIO(raw)))
        elif fmt in ("excel","xlsx"):
            return spark.createDataFrame(pd.read_excel(io.BytesIO(raw)))
        else:
            raise ValueError(f"Unsupported HTTP format: {fmt}")
    else:
        if fmt == "csv":
            return spark.read.option("header","true").option("inferSchema","true").csv(source_url)
        elif fmt == "json":
            return spark.read.option("multiLine","true").json(source_url)
        elif fmt == "parquet":
            return spark.read.parquet(source_url)
        elif fmt == "delta":
            return spark.read.format("delta").load(source_url)
        else:
            raise ValueError(f"Unsupported format: {fmt}")

def ensure_schema(catalog, schema):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

def write_full(df, full_table):
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(full_table)

def write_merge(df, full_table, merge_keys_str):
    """MERGE into existing table using merge keys."""
    keys = [k.strip() for k in merge_keys_str.split(",")]
    match_cond = " AND ".join([f"tgt.{k} = src.{k}" for k in keys])

    # Write to temp view
    tmp_view = f"_tmp_merge_{full_table.replace('.','_')}"
    df.createOrReplaceTempView(tmp_view)

    # Create table if not exists
    spark.sql(f"CREATE TABLE IF NOT EXISTS {full_table} USING DELTA AS SELECT * FROM {tmp_view} WHERE 1=0")

    # Non-key columns for update
    all_cols  = df.columns
    upd_cols  = [c for c in all_cols if c not in keys]
    upd_set   = ", ".join([f"tgt.{c} = src.{c}" for c in upd_cols])
    ins_cols  = ", ".join(all_cols)
    ins_vals  = ", ".join([f"src.{c}" for c in all_cols])

    spark.sql(f"""
        MERGE INTO {full_table} AS tgt
        USING {tmp_view} AS src
        ON {match_cond}
        WHEN MATCHED THEN UPDATE SET {upd_set}
        WHEN NOT MATCHED THEN INSERT ({ins_cols}) VALUES ({ins_vals})
    """)

def write_table(df, catalog, schema, table, load_type, merge_keys=None):
    ensure_schema(catalog, schema)
    full = f"{catalog}.{schema}.{table}"
    lt   = load_type.strip().upper()

    if lt == "FULL":
        write_full(df, full)
    elif lt in ("INCREMENTAL","APPEND"):
        df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(full)
    elif lt == "MERGE" and merge_keys:
        write_merge(df, full, merge_keys)
    else:
        write_full(df, full)

    return spark.table(full).count()

def write_audit(group_id, table_name, layer, status, message):
    try:
        msg = message.replace("'","''")[:500]
        spark.sql(f"""
            INSERT INTO {CATALOG}.admin.audit_log
            VALUES ('{group_id}', '{table_name}', '{status}', '{layer}: {msg}', current_timestamp())
        """)
    except Exception as e:
        print(f"[AUDIT FAIL] {e}")

In [0]:
# ── Read header ───────────────────────────────────────────────

header_df = spark.sql(f"""
    SELECT * FROM {CATALOG}.admin.data_flow_control_header
    WHERE DATA_FLOW_GROUP_ID = '{group_id}' AND IS_ACTIVE = 'Y'
""")

if header_df.count() == 0:
    raise Exception(f"No active header for GROUP_ID='{group_id}'")

header   = header_df.first()
etl_layer = (header['ETL_LAYER'] or 'L0').upper()
print(f"ETL_LAYER from config: {etl_layer}")

In [0]:
# ══════════════════════════════════════════
# L0 — BRONZE (Raw ingestion)
# ══════════════════════════════════════════

def run_l0():
    print("\n" + "═"*50)
    print("  L0 — BRONZE INGESTION")
    print("═"*50)

    rows = spark.sql(f"""
        SELECT SOURCE_URL, SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE, FILE_FORMAT,
               INPUT_FILE_FORMAT, LOAD_TYPE, DELIMETER
        FROM {CATALOG}.admin.data_flow_l0_detail
        WHERE DATA_FLOW_GROUP_ID = '{group_id}' AND IS_ACTIVE = 'Y'
    """).collect()

    if not rows:
        raise Exception(f"No active L0 detail rows for {group_id}")

    all_ok = True
    for row in rows:
        source_url   = row['SOURCE_URL']
        fmt          = row['INPUT_FILE_FORMAT'] or row['FILE_FORMAT'] or 'csv'
        target_schema = row['TARGET_SCHEMA'] or 'bronze'
        target_table  = row['TARGET_TABLE']
        load_type     = row['LOAD_TYPE'] or 'FULL'
        full_name     = f"{CATALOG}.{target_schema}.{target_table}"

        print(f"\n▶  {full_name} | {fmt} | {load_type}")
        start = datetime.now()

        try:
            df = read_source(source_url, fmt)
            df = df.withColumn("_etl_group_id", F.lit(group_id)) \
                   .withColumn("_etl_layer",    F.lit("L0")) \
                   .withColumn("_etl_load_ts",  F.current_timestamp())

            count = write_table(df, CATALOG, target_schema, target_table, load_type)
            print(f"   ✅ {count:,} rows → {full_name}")
            write_audit(group_id, target_table, "L0", "SUCCESS", f"{count} rows loaded")

        except Exception as e:
            all_ok = False
            msg = traceback.format_exc()
            print(f"   ❌ FAILED: {e}")
            write_audit(group_id, target_table, "L0", "FAILED", msg)
        finally:
            print(f"   Duration: {round((datetime.now()-start).total_seconds(),1)}s")

    if not all_ok:
        raise Exception(f"L0 had failures for {group_id}")

In [0]:
# ══════════════════════════════════════════
# L1 — SILVER (Cleaned / conformed)
# ══════════════════════════════════════════

def run_l1():
    print("\n" + "═"*50)
    print("  L1 — SILVER TRANSFORMATION")
    print("═"*50)

    rows = spark.sql(f"""
        SELECT SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE,
               LOAD_TYPE, TRANSFORMATION_QUERY, MERGE_KEYS, PARTITION_BY
        FROM {CATALOG}.admin.data_flow_l1_detail
        WHERE DATA_FLOW_GROUP_ID = '{group_id}' AND IS_ACTIVE = 'Y'
    """).collect()

    if not rows:
        print("  No L1 config found — skipping Silver layer")
        return

    all_ok = True
    for row in rows:
        src_schema  = row['SOURCE_OBJ_SCHEMA'] or 'bronze'
        src_table   = row['SOURCE_OBJ_NAME']
        tgt_schema  = row['TARGET_SCHEMA'] or 'silver'
        tgt_table   = row['TARGET_TABLE']
        load_type   = row['LOAD_TYPE'] or 'MERGE'
        trans_query = row['TRANSFORMATION_QUERY']
        merge_keys  = row['MERGE_KEYS']
        full_name   = f"{CATALOG}.{tgt_schema}.{tgt_table}"

        print(f"\n▶  {CATALOG}.{src_schema}.{src_table} → {full_name}")
        start = datetime.now()

        try:
            if trans_query:
                # Replace schema references to include catalog
                final_query = trans_query \
                    .replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.") \
                    .replace(f"FROM {src_table}", f"FROM {CATALOG}.{src_schema}.{src_table}")
                df = spark.sql(final_query)
            else:
                df = spark.table(f"{CATALOG}.{src_schema}.{src_table}")

            df = df.withColumn("_etl_group_id", F.lit(group_id)) \
                   .withColumn("_etl_layer",    F.lit("L1")) \
                   .withColumn("_etl_load_ts",  F.current_timestamp())

            count = write_table(df, CATALOG, tgt_schema, tgt_table, load_type, merge_keys)
            print(f"   ✅ {count:,} rows → {full_name}")
            write_audit(group_id, tgt_table, "L1", "SUCCESS", f"{count} rows")

        except Exception as e:
            all_ok = False
            msg = traceback.format_exc()
            print(f"   ❌ FAILED: {e}")
            write_audit(group_id, tgt_table, "L1", "FAILED", msg)
        finally:
            print(f"   Duration: {round((datetime.now()-start).total_seconds(),1)}s")

    if not all_ok:
        raise Exception(f"L1 had failures for {group_id}")

In [0]:
# ══════════════════════════════════════════
# L2 — GOLD (Aggregated / business ready)
# ══════════════════════════════════════════

def run_l2():
    print("\n" + "═"*50)
    print("  L2 — GOLD AGGREGATION")
    print("═"*50)

    rows = spark.sql(f"""
        SELECT SOURCE_OBJ_SCHEMA, SOURCE_OBJ_NAME,
               TARGET_SCHEMA, TARGET_TABLE,
               LOAD_TYPE, TRANSFORMATION_QUERY, MERGE_KEYS, PARTITION_BY
        FROM {CATALOG}.admin.data_flow_l2_detail
        WHERE DATA_FLOW_GROUP_ID = '{group_id}' AND IS_ACTIVE = 'Y'
    """).collect()

    if not rows:
        print("  No L2 config found — skipping Gold layer")
        return

    all_ok = True
    for row in rows:
        src_schema  = row['SOURCE_OBJ_SCHEMA'] or 'silver'
        src_table   = row['SOURCE_OBJ_NAME']
        tgt_schema  = row['TARGET_SCHEMA'] or 'gold'
        tgt_table   = row['TARGET_TABLE']
        load_type   = row['LOAD_TYPE'] or 'FULL'
        trans_query = row['TRANSFORMATION_QUERY']
        merge_keys  = row['MERGE_KEYS']
        full_name   = f"{CATALOG}.{tgt_schema}.{tgt_table}"

        print(f"\n▶  {CATALOG}.{src_schema}.{src_table} → {full_name}")
        start = datetime.now()

        try:
            if trans_query:
                final_query = trans_query \
                    .replace(f"{src_schema}.", f"{CATALOG}.{src_schema}.") \
                    .replace(f"FROM {src_table}", f"FROM {CATALOG}.{src_schema}.{src_table}")
                df = spark.sql(final_query)
            else:
                df = spark.table(f"{CATALOG}.{src_schema}.{src_table}")

            df = df.withColumn("_etl_group_id", F.lit(group_id)) \
                   .withColumn("_etl_layer",    F.lit("L2")) \
                   .withColumn("_etl_load_ts",  F.current_timestamp())

            count = write_table(df, CATALOG, tgt_schema, tgt_table, load_type, merge_keys)
            print(f"   ✅ {count:,} rows → {full_name}")
            write_audit(group_id, tgt_table, "L2", "SUCCESS", f"{count} rows")

        except Exception as e:
            all_ok = False
            msg = traceback.format_exc()
            print(f"   ❌ FAILED: {e}")
            write_audit(group_id, tgt_table, "L2", "FAILED", msg)
        finally:
            print(f"   Duration: {round((datetime.now()-start).total_seconds(),1)}s")

    if not all_ok:
        raise Exception(f"L2 had failures for {group_id}")

In [0]:
# ── Execute layers based on config ────────────────────────────

if run_layer == "L0":
    run_l0()
elif run_layer == "L1":
    run_l1()
elif run_layer == "L2":
    run_l2()
else:
    # fallback: run all if suffix not recognised
    run_l0()
    run_l1()
    run_l2()

In [0]:
# ── Final summary ─────────────────────────────────────────────

print(f"\n{'═'*50}")
print(f"  COMPLETE — GROUP_ID: {group_id}")
print(f"  Layer(s) run: {run_layer}")
print(f"  Check audit: SELECT * FROM {CATALOG}.admin.audit_log")
print(f"               WHERE DATA_FLOW_GROUP_ID = '{group_id}'")
print(f"               ORDER BY LOAD_TS DESC")
print(f"{'═'*50}")